# Call Center Volume Forecasting

**Project 10 of 10** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **daily call center call volumes** using a public call-center dataset. The goal is to predict how many calls a center will receive each day, enabling proper staffing and workforce scheduling.

| Attribute | Value |
|---|---|
| **Target** | `calls` (daily call count) |
| **Frequency** | Daily (`D`) |
| **Seasonal period** | 7 (weekly cycle) |
| **Primary TS library** | MLForecast |
| **Dataset** | Kaggle `call-center-data` or synthetic |

### Why MLForecast?
Call center volumes exhibit strong day-of-week patterns (weekday peaks, weekend troughs) and gradual trends. Gradient boosting with lag features captures these patterns effectively, and MLForecast handles the recursive forecasting loop natively.

## Learning Objectives

1. Load and validate call center records
2. Aggregate to **daily call volume** time series
3. Explore **day-of-week**, **monthly**, and **trend** patterns
4. Build **naive** and **seasonal naive (7-day)** baselines
5. Benchmark via **LazyPredict** on lag features
6. Run **FLAML AutoML** for model search
7. Train **MLForecast** models (LightGBM, XGBoost, Ridge)
8. Evaluate with **MAE, RMSE, MAPE** and compare against baselines
9. Perform error analysis and identify challenging periods
10. Discuss **workforce management** applications

## Problem Statement

Given historical daily call volumes for a contact center, **forecast the next 14 days of call volume**.

Accurate call volume forecasting enables:
- **Workforce scheduling** — staff the right number of agents each day
- **Service level management** — maintain target answer rates (e.g., 80/20 rule)
- **Cost optimization** — avoid overstaffing (wasted labor) and understaffing (abandoned calls)
- **Capacity planning** — scale IVR systems, phone lines, and queue infrastructure

## Why This Project Matters

- **Contact centers employ 17M+ people globally** — staffing is the #1 cost driver
- **1% overstaffing** at scale wastes millions in unnecessary labor costs
- **1% understaffing** causes abandoned calls, poor customer satisfaction, and revenue loss
- **Forecast accuracy directly drives business outcomes** — a 5% MAPE improvement can save 3-5% of total labor cost
- This is one of the most common **real-world time series** problems in operations management

## Dataset Overview

We use a synthetic call center dataset that mimics real-world patterns:
- **~365 days** of daily call records
- **Weekday/weekend pattern**: Higher volume on weekdays, lower on weekends
- **Monthly seasonality**: Some months busier than others
- **Gradual trend**: Slight growth over time
- **Random noise**: Realistic day-to-day variability

If a Kaggle dataset is available, we download it; otherwise, we generate realistic synthetic data.

## Dataset Source & License Notes

- **Source**: Synthetic generation or Kaggle public dataset
- **License**: Public domain / educational use
- **Note**: If using Kaggle data, competition/dataset rules apply

## Environment Setup

Install all required packages. This cell is idempotent — safe to rerun.

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "mlforecast", "lightgbm", "xgboost",
    "statsmodels", "scipy", "window-ops",
]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
import lightgbm as lgb
import xgboost as xgb
from window_ops.rolling import rolling_mean, rolling_std

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
np.random.seed(42)

print("All imports successful.")

## Configuration & Constants

In [ ]:
PROJECT_NAME = "Call Center Volume Forecasting"

TARGET  = "calls"
FREQ    = "D"

SEASON_LENGTH    = 7     # weekly cycle
FORECAST_HORIZON = 14    # predict next 2 weeks
TEST_SIZE        = FORECAST_HORIZON
VAL_SIZE         = FORECAST_HORIZON
RANDOM_STATE     = 42
FLAML_BUDGET     = 120   # seconds

print(f"Project : {PROJECT_NAME}")
print(f"Target  : {TARGET}  |  Freq: {FREQ}  |  Season: {SEASON_LENGTH}")
print(f"Horizon : {FORECAST_HORIZON} days")

## Dataset Download or Loading

We first try to load a real call center dataset. If unavailable, we generate a realistic synthetic dataset with:
- Strong weekday/weekend pattern (weekdays ~500-700, weekends ~200-350)
- Monthly seasonality (January & March busier)
- Slight upward trend
- Random noise

In [ ]:
# Try Kaggle first
try:
    import kagglehub
    data_path = pathlib.Path(kagglehub.dataset_download("kyanyoga/callcenter"))
    csv_files = list(data_path.rglob("*.csv"))
    if csv_files:
        raw = pd.read_csv(csv_files[0])
        print(f"Loaded Kaggle dataset: {raw.shape}")
        has_kaggle = True
    else:
        has_kaggle = False
except Exception:
    has_kaggle = False

if not has_kaggle:
    print("Kaggle dataset unavailable — generating realistic synthetic data.")
    # Generate 2 years of daily call data
    dates = pd.date_range("2022-01-01", periods=730, freq="D")
    np.random.seed(RANDOM_STATE)
    
    # Base volume per day of week (Mon=0 ... Sun=6)
    dow_pattern = {0: 620, 1: 650, 2: 640, 3: 630, 4: 580, 5: 310, 6: 250}
    base = np.array([dow_pattern[d.dayofweek] for d in dates], dtype=float)
    
    # Monthly seasonality
    month_effect = {1: 1.08, 2: 1.02, 3: 1.06, 4: 1.00, 5: 0.97, 6: 0.93,
                    7: 0.90, 8: 0.92, 9: 1.03, 10: 1.05, 11: 1.01, 12: 0.95}
    monthly = np.array([month_effect[d.month] for d in dates])
    
    # Slight upward trend
    trend = np.linspace(1.0, 1.08, len(dates))
    
    # Random noise
    noise = np.random.normal(0, 30, len(dates))
    
    calls = (base * monthly * trend + noise).clip(50).astype(int)
    raw = pd.DataFrame({"date": dates, "calls": calls})
    print(f"Generated synthetic data: {raw.shape}")

raw.head()

## Data Validation Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)

# Standardize column names
if "date" not in raw.columns:
    date_cols = [c for c in raw.columns if "date" in c.lower() or "time" in c.lower()]
    if date_cols:
        raw = raw.rename(columns={date_cols[0]: "date"})

if "calls" not in raw.columns:
    # If we have individual call records, aggregate to daily
    if "date" in raw.columns:
        raw["date"] = pd.to_datetime(raw["date"])
        daily = raw.groupby(raw["date"].dt.date).size().reset_index(name="calls")
        daily["date"] = pd.to_datetime(daily["date"])
        raw = daily
    else:
        num_cols = raw.select_dtypes(include=[np.number]).columns
        if len(num_cols) > 0:
            raw = raw.rename(columns={num_cols[0]: "calls"})

raw["date"] = pd.to_datetime(raw["date"])
raw = raw.sort_values("date").reset_index(drop=True)

print(f"  Rows         : {len(raw):,}")
print(f"  Date range   : {raw['date'].min().date()} to {raw['date'].max().date()}")
print(f"  Days span    : {(raw['date'].max() - raw['date'].min()).days}")
print(f"  NaN in calls : {raw['calls'].isna().sum()}")
print(f"  Duplicates   : {raw.duplicated(subset=['date']).sum()}")
print(f"  Negative vals: {(raw['calls'] < 0).sum()}")
print("\nValidation complete.")

## Exploratory Data Analysis

In [ ]:
# Full time series
fig = go.Figure()
fig.add_trace(go.Scatter(x=raw["date"], y=raw["calls"],
                         name="Daily Calls", line=dict(width=1)))
fig.add_trace(go.Scatter(x=raw["date"],
                         y=raw["calls"].rolling(7).mean(),
                         name="7-day MA", line=dict(color="red", width=2)))
fig.add_trace(go.Scatter(x=raw["date"],
                         y=raw["calls"].rolling(30).mean(),
                         name="30-day MA", line=dict(color="green", width=2, dash="dash")))
fig.update_layout(title="Call Center — Daily Call Volume",
                  xaxis_title="Date", yaxis_title="Calls",
                  template="plotly_white")
fig.show()

In [ ]:
# Day-of-week boxplot
raw["dow"] = raw["date"].dt.day_name()
fig = px.box(raw, x="dow", y="calls",
             category_orders={"dow": ["Monday","Tuesday","Wednesday","Thursday",
                                       "Friday","Saturday","Sunday"]},
             title="Call Volume by Day of Week")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Monthly pattern
raw["month"] = raw["date"].dt.month_name()
monthly_avg = raw.groupby(raw["date"].dt.month)["calls"].mean().reset_index()
monthly_avg.columns = ["Month", "Avg Calls"]
fig = px.bar(monthly_avg, x="Month", y="Avg Calls",
             title="Average Daily Calls by Month")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Seasonal decomposition
ts = raw.set_index("date")["calls"].asfreq("D")
ts = ts.interpolate() if ts.isna().any() else ts
decomp = seasonal_decompose(ts, model="additive", period=SEASON_LENGTH)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Weekly Seasonal")
decomp.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout()
plt.show()

## Target Analysis

Understanding the statistical properties of the call volume target.

In [ ]:
print("Call Volume Statistics:")
print(raw["calls"].describe().to_string())
print(f"\nSkewness : {raw["calls"].skew():.3f}")
print(f"Kurtosis : {raw["calls"].kurtosis():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(raw["calls"], bins=40, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of Daily Call Volume")
axes[1].boxplot(raw["calls"].dropna())
axes[1].set_title("Box Plot")
pd.plotting.lag_plot(raw["calls"], lag=7, ax=axes[2])
axes[2].set_title("Lag-7 Plot (Weekly)")
plt.tight_layout()
plt.show()

In [ ]:
# Stationarity test
adf = adfuller(ts.dropna())
print(f"ADF statistic: {adf[0]:.4f}")
print(f"p-value      : {adf[1]:.6f}")
print(f"Result       : {'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'}")

## Train / Validation / Test Split Strategy

Strict temporal split — last 14 days for test, preceding 14 for validation, rest for training. No shuffling, no leakage.

In [ ]:
ts_df = raw[["date", "calls"]].rename(columns={"date": "ds", "calls": "y"}).copy()
n = len(ts_df)

ts_train = ts_df.iloc[:n - TEST_SIZE - VAL_SIZE].copy()
ts_val   = ts_df.iloc[n - TEST_SIZE - VAL_SIZE : n - TEST_SIZE].copy()
ts_test  = ts_df.iloc[n - TEST_SIZE:].copy()

print(f"Train : {len(ts_train)} days ({ts_train['ds'].min().date()} to {ts_train['ds'].max().date()})")
print(f"Val   : {len(ts_val)} days ({ts_val['ds'].min().date()} to {ts_val['ds'].max().date()})")
print(f"Test  : {len(ts_test)} days ({ts_test['ds'].min().date()} to {ts_test['ds'].max().date()})")

assert ts_train["ds"].max() < ts_val["ds"].min(), "Train/Val overlap!"
assert ts_val["ds"].max() < ts_test["ds"].min(), "Val/Test overlap!"
print("No temporal overlap — split is clean.")

ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train"))
fig.add_trace(go.Scatter(x=ts_val["ds"], y=ts_val["y"], name="Validation", line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Test", line=dict(color="red")))
fig.update_layout(title="Temporal Split Visualization", template="plotly_white")
fig.show()

## Preprocessing Strategy

- **No scaling** for tree-based models (LightGBM, XGBoost)
- **Ridge regression** benefits from scaled features but sklearn's Ridge handles it internally
- **Missing dates** filled via interpolation
- **Lag features** automatically prevent leakage via `.shift()`

## Feature Engineering

We create temporal lag features and calendar indicators for the tabular modeling approaches.

In [ ]:
def make_lag_features(df):
    """Create lag, rolling, and calendar features."""
    out = df.copy()
    for lag in [1, 7, 14, 21, 28]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    for window in [7, 14, 28]:
        out[f"rolling_mean_{window}"] = out["y"].shift(1).rolling(window).mean()
        out[f"rolling_std_{window}"]  = out["y"].shift(1).rolling(window).std()
    out["dayofweek"] = out["ds"].dt.dayofweek
    out["month"]     = out["ds"].dt.month
    out["day"]       = out["ds"].dt.day
    out["is_weekend"] = (out["ds"].dt.dayofweek >= 5).astype(int)
    return out

feat = make_lag_features(ts_df).dropna().reset_index(drop=True)
feature_cols = [c for c in feat.columns if c not in ["ds", "y"]]

train_cutoff = ts_train["ds"].max()
val_cutoff = ts_val["ds"].max()

f_train = feat[feat["ds"] <= train_cutoff]
f_val   = feat[(feat["ds"] > train_cutoff) & (feat["ds"] <= val_cutoff)]
f_test  = feat[feat["ds"] > val_cutoff]

X_train, y_train = f_train[feature_cols], f_train["y"]
X_val, y_val     = f_val[feature_cols], f_val["y"]
X_test, y_test   = f_test[feature_cols], f_test["y"]
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

## Baseline Model

Simple baselines for call center forecasting — if advanced models can't beat these, they're not worth the complexity.

In [ ]:
def calc_metrics(actual, predicted, name="Model"):
    actual = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.any() else np.nan
    return {"Model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE": round(mape, 2)}

results = []
actual_test = ts_test["y"].values

# Naive
results.append(calc_metrics(actual_test,
    np.full(TEST_SIZE, ts_trainval["y"].iloc[-1]), "Naive (last value)"))

# Seasonal Naive (repeat last week pattern, tiled for 14 days)
seasonal_pattern = ts_trainval["y"].iloc[-SEASON_LENGTH:].values
results.append(calc_metrics(actual_test,
    np.tile(seasonal_pattern, 3)[:TEST_SIZE], "Seasonal Naive (7-day)"))

# Moving Average
results.append(calc_metrics(actual_test,
    np.full(TEST_SIZE, ts_trainval["y"].iloc[-7:].mean()), "MA(7)"))

# Same-weekday-last-week
last_2w = ts_trainval["y"].iloc[-14:].values
results.append(calc_metrics(actual_test, last_2w[:TEST_SIZE], "Same Day Last 2 Weeks"))

print("Baseline Results:")
print(pd.DataFrame(results).to_string(index=False))

## LazyPredict Benchmark

Quick tabular regression benchmark on lag features — identifies promising model families.

In [ ]:
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=RANDOM_STATE)
    lazy_models, _ = lazy.fit(X_train, X_val, y_train, y_val)
    print("LazyPredict — Top 15 Models:")
    print(lazy_models.head(15).to_string())
    for i, (name, row) in enumerate(lazy_models.head(3).iterrows()):
        results.append({
            "Model": f"LP: {name}",
            "MAE": round(row.get("MAE", 0), 2),
            "RMSE": round(row.get("RMSE", 0), 2),
            "MAPE": np.nan
        })
except Exception as e:
    print(f"LazyPredict failed: {e}")

## FLAML AutoML Run

In [ ]:
try:
    flaml_model = AutoML()
    flaml_model.fit(
        X_train=X_trainval, y_train=y_trainval,
        task="regression", time_budget=FLAML_BUDGET,
        metric="rmse", verbose=0, seed=RANDOM_STATE
    )
    flaml_preds = flaml_model.predict(X_test)
    results.append(calc_metrics(
        actual_test[:len(flaml_preds)], flaml_preds,
        f"FLAML ({flaml_model.best_estimator})"
    ))
    print(f"FLAML best estimator: {flaml_model.best_estimator}")
except Exception as e:
    print(f"FLAML failed: {e}")

## Top 3 Model Selection

In [ ]:
results_df_mid = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("Current Ranking:")
print(results_df_mid.to_string(index=False))

## MLForecast — Dedicated Time-Series Models

**Why MLForecast for call centers?** Call volume has strong day-of-week seasonality (weekday vs weekend). MLForecast's lag-7 feature and day-of-week encoding capture this directly.

In [ ]:
mlf_data = ts_trainval.copy()
mlf_data["unique_id"] = "call_center"

mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=31,
            random_state=RANDOM_STATE, verbosity=-1
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            random_state=RANDOM_STATE, verbosity=0
        ),
        "Ridge": Ridge(alpha=1.0),
    },
    freq="D",
    lags=[1, 7, 14, 21, 28],
    lag_transforms={
        1: [(rolling_mean, 7), (rolling_mean, 14), (rolling_std, 7)],
        7: [(rolling_mean, 4)],
    },
    date_features=["dayofweek", "month", "day"],
)

mlf.fit(mlf_data)
mlf_preds = mlf.predict(h=FORECAST_HORIZON)
print(f"MLForecast predictions: {mlf_preds.shape}")
mlf_preds.head()

In [ ]:
for model_name in ["LightGBM", "XGBoost", "Ridge"]:
    if model_name in mlf_preds.columns:
        pred_vals = mlf_preds[model_name].values[:TEST_SIZE]
        r = calc_metrics(actual_test, pred_vals, f"MLF: {model_name}")
        results.append(r)
        print(f"{model_name}: MAE={r['MAE']}, RMSE={r['RMSE']}, MAPE={r['MAPE']}%")

## Final Training & Evaluation of Top 3

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("=" * 60)
print("ALL MODELS — FINAL RANKING")
print("=" * 60)
print(results_df.to_string(index=False))

top3 = results_df.head(3)
print(f"\n{'=' * 60}")
print("TOP 3 MODELS:")
print(f"{'=' * 60}")
print(top3.to_string(index=False))

fig = px.bar(results_df, x="Model", y="RMSE",
             title="Model Comparison — RMSE (lower is better)",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

## Forecast vs Actual — Test Period

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ts_test["ds"], y=actual_test,
    name="Actual", line=dict(color="black", width=3)
))

# Baselines
sn_pred = np.tile(seasonal_pattern, 3)[:TEST_SIZE]
fig.add_trace(go.Scatter(
    x=ts_test["ds"], y=sn_pred,
    name="Seasonal Naive", line=dict(color="gray", dash="dot")
))

# MLForecast models
for model_name in ["LightGBM", "XGBoost", "Ridge"]:
    if model_name in mlf_preds.columns:
        fig.add_trace(go.Scatter(
            x=ts_test["ds"], y=mlf_preds[model_name].values[:TEST_SIZE],
            name=f"MLF: {model_name}", line=dict(dash="dash")
        ))

fig.update_layout(
    title="Call Volume Forecast vs Actual — Test Period (14 days)",
    xaxis_title="Date", yaxis_title="Daily Calls",
    template="plotly_white"
)
fig.show()

## Error Analysis

Identifying when and why the model fails.

In [ ]:
best_model = "LightGBM" if "LightGBM" in mlf_preds.columns else mlf_preds.columns[-1]
best_pred = mlf_preds[best_model].values[:TEST_SIZE]
errors = actual_test - best_pred
pct_errors = np.where(actual_test != 0, errors / actual_test * 100, 0)

# Add day-of-week context
test_days = ts_test["ds"].dt.day_name().values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(errors, bins=15, edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", ls="--")
axes[0].set_title("Error Distribution")

axes[1].bar(range(TEST_SIZE), np.abs(pct_errors))
axes[1].set_title("|% Error| by Day")
axes[1].set_xticks(range(TEST_SIZE))
axes[1].set_xticklabels(test_days, rotation=45, ha="right", fontsize=8)

axes[2].scatter(actual_test, best_pred, alpha=0.7)
min_v, max_v = min(actual_test.min(), best_pred.min()), max(actual_test.max(), best_pred.max())
axes[2].plot([min_v, max_v], [min_v, max_v], "r--")
axes[2].set_title("Actual vs Predicted")
axes[2].set_xlabel("Actual")
axes[2].set_ylabel("Predicted")
plt.tight_layout()
plt.show()

print(f"Mean Bias     : {errors.mean():+.1f} calls")
print(f"Mean |Error|  : {np.abs(errors).mean():.1f} calls")
print(f"Max |Error|   : {np.abs(errors).max():.0f} calls")
print(f"Mean |%Error| : {np.abs(pct_errors).mean():.1f}%")

## Interpretation & Business Insight

1. **Day-of-week is the dominant signal**: Weekday volumes are 2-3x weekend volumes — lag-7 captures this directly
2. **Seasonal naive is a strong baseline**: In call centers, last week's pattern often repeats almost exactly
3. **ML models add value by capturing trend and month effects**: Pure seasonal naive misses gradual volume changes
4. **Staffing implications**: A MAPE of 5% on 600 daily calls = ±30 calls = ±3-4 agents in a typical center
5. **Weekend prediction is easier**: Lower, more stable volumes → lower absolute errors
6. **Monday/Friday transitions are hardest**: Volume ramps between weekend and weekday create the largest errors

## Limitations

1. **Synthetic data**: The patterns are realistic but not from a real contact center
2. **No intra-day detail**: Half-hourly or 15-minute intervals are standard in WFM
3. **No external drivers**: Marketing campaigns, outages, product launches drive spikes
4. **Single center**: Multi-site operations need hierarchical forecasting
5. **No holiday handling**: Major holidays radically alter call patterns
6. **No skill-based routing**: Different call types have different volume patterns

## How to Improve This Project

1. **Real WFM data**: Use actual call center data from workforce management systems
2. **Intra-day forecasting**: Forecast at 15-minute or 30-minute intervals
3. **Holiday features**: Add national/regional holiday indicators
4. **Hierarchical models**: Forecast by call type (billing, tech support, sales)
5. **Erlang-C integration**: Convert call volume forecast to required staffing levels
6. **Ensemble**: Average top models for more robust forecasts
7. **Conformal prediction**: Generate prediction intervals for risk-aware staffing

## Production Considerations

1. **Daily retraining**: Update model each morning with previous day's actuals
2. **WFM integration**: Feed forecasts into Erlang-C calculators for staffing plans
3. **Intra-day adjustment**: Re-forecast during the day as actual volumes deviate
4. **Exception handling**: Override model on known holidays, outages, promotions
5. **Monitoring**: Track forecast accuracy weekly; alert if MAPE > threshold
6. **Fallback**: Maintain seasonal naive as backup if model fails
7. **Audit trail**: Log all forecasts and actuals for compliance and improvement

## Common Mistakes to Avoid

1. **Ignoring day-of-week**: The strongest signal in call center data
2. **Random train/test splits**: Always temporal for time series
3. **Forecasting volume without considering handle time**: Staffing = f(volume, AHT, service level)
4. **Using MAPE with near-zero weekend volumes**: Can inflate metric artificially
5. **Not accounting for holidays**: Christmas Day ≠ normal Tuesday
6. **Over-fitting to recent trend**: Trends in call centers can reverse suddenly
7. **Ignoring data quality**: Abandoned calls, callbacks, and transfers affect counts

## Mini Challenge / Exercises

1. **Intra-day forecast**: Resample to hourly and forecast 24h of call volume
2. **Weekend vs weekday models**: Train separate models for weekdays and weekends
3. **Holiday injection**: Add a binary holiday column and observe impact
4. **Erlang-C calculation**: Convert your 14-day forecast to required agents (assume AHT=300s, service level=80/20)
5. **Rolling CV**: Evaluate on 4+ temporal windows for robust performance estimates

## Final Summary & Key Takeaways

### What We Did
- Built a realistic call center dataset with weekday/weekend patterns, monthly seasonality, and trend
- Explored day-of-week and monthly patterns in call volumes
- Built baselines (naive, seasonal naive, moving average, same-day-last-week)
- Benchmarked regression models via **LazyPredict** on lag features
- Ran **FLAML AutoML** for model search
- Trained **MLForecast** models (LightGBM, XGBoost, Ridge)
- Compared all models and performed error analysis

### Key Takeaways
1. **Weekly cycle (7-day) is the dominant pattern** — weekday/weekend is everything
2. **Seasonal naive is surprisingly strong** — last week repeats well in stable centers
3. **ML models add value** by capturing trend, monthly effects, and reducing variance
4. **A 5% MAPE** translates to ±30 calls on a 600-call day = ±3-4 agents
5. **Production WFM** needs intra-day granularity, holiday handling, and Erlang-C integration

---
*Notebook generated as part of a time-series forecasting portfolio.*
*For educational purposes only — always validate with real data before production use.*